# **Lab 9 - Agency & Autonomy Part 2**
<p>COMP4146/7125 Prompt Engineering for Generative AI<br/>Semester 2, 2025/26, Dr. Shichao Ma, HKBU</p>


## Lab Overview

This lab implements the **Shopify plug-in marketing workflow** from the lecture in a generic, reusable way:

`website URL -> fetch HTML file -> extract signals -> summarize store -> propose plug-in -> plan email -> draft final email`

All live model-based cells reuse one backend throughout: **`minimax-m2.5:cloud`** through Ollama.

**By the end of this notebook, you should be able to:**
1. build a workflow that starts from a storefront URL instead of from manually prepared store notes;
2. fetch and save real storefront HTML files with Python;
3. pass stable intermediate artifacts between workflow tasks;
4. inspect uncertain or imperfect outputs instead of pretending the workflow is fully deterministic;
5. produce a final outreach email from a generic storefront workflow.


## Setup

This notebook combines:
- deterministic Python code for fetching, parsing, and orchestration;
- live Ollama calls using **`minimax-m2.5:cloud`** for reasoning-heavy tasks.

Before running the notebook, make sure:
- `ollama serve` is running;
- you already signed in with `ollama signin`;
- the Ollama Python package is available;
- the notebook is allowed to access public websites.

The workflow is intentionally **not fully rigid**. Some steps involve uncertainty, especially when storefront HTML is noisy or when the model proposes a creative plug-in concept. That is acceptable here because the tutorial is about workflow design, inspection, and iteration.


In [1]:
!pip install requests beautifulsoup4 ollama


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## Input Configuration

The workflow accepts any public storefront URLs.

You can provide inputs by editing the `WORKFLOW_INPUTS` dictionary in the next cell



In [2]:
from __future__ import annotations

import json
import re
from collections import Counter
from copy import deepcopy
from pathlib import Path
from textwrap import dedent
from urllib.parse import urlparse

import requests
from bs4 import BeautifulSoup


def show_json(obj):
    # Pretty-print nested workflow artifacts so intermediate outputs are easy to inspect.
    print(json.dumps(obj, indent=2, ensure_ascii=True))


def short(text, limit=160):
    # Compact previews keep notebook outputs readable even when HTML is large.
    text = " ".join(str(text).split())
    return text if len(text) <= limit else text[: limit - 3] + "..."


def contains_placeholder_text(text):
    text = str(text or "").lower()
    return any(marker in text for marker in ["[insert", "[link", "placeholder", "lorem ipsum"])


def to_builtin(obj):
    # Convert model responses into plain Python values before downstream parsing.
    if hasattr(obj, "model_dump"):
        return to_builtin(obj.model_dump())
    if isinstance(obj, dict):
        return {key: to_builtin(value) for key, value in obj.items()}
    if isinstance(obj, list):
        return [to_builtin(value) for value in obj]
    return obj


def split_items(value):
    # Normalize strings or lists into short cleaned lists.
    if isinstance(value, list):
        raw_items = value
    elif isinstance(value, str):
        raw_items = re.split(r",|;|\n", value)
    else:
        raw_items = []
    return [str(item).strip() for item in raw_items if str(item).strip()]


In [3]:
WORKFLOW_INPUTS = {
    # Paste one or more public storefront URLs here.
    "storefront_urls": ["https://flybyjing.com/", "https://www.allbirds.com/"],
    # Keep sender details configurable so the drafted email is reusable.
    "sender_profile": {
        "company_name": "Partnerships Team",
        "signature": "Partnerships Team",
    },
    # "best_reviewed" prefers outputs that pass the notebook review heuristics.
    "selection_policy": "best_reviewed",
}

storefront_urls = list(WORKFLOW_INPUTS.get("storefront_urls") or [])

if not isinstance(storefront_urls, list):
    raise TypeError("storefront_urls must be a list of public website URLs.")

storefront_urls = [str(url).strip() for url in storefront_urls if str(url).strip()]
if not storefront_urls:
    raise ValueError(
        "Provide at least one storefront URL in WORKFLOW_INPUTS['storefront_urls'] "
    )

sender_profile = dict(WORKFLOW_INPUTS.get("sender_profile") or {})
selection_policy = str(WORKFLOW_INPUTS.get("selection_policy") or "best_reviewed").strip()
if selection_policy not in {"best_reviewed", "first"}:
    selection_policy = "best_reviewed"

show_json(
    {
        "storefront_urls": storefront_urls,
        "sender_profile": sender_profile,
        "selection_policy": selection_policy,
    }
)


{
  "storefront_urls": [
    "https://flybyjing.com/",
    "https://www.allbirds.com/"
  ],
  "sender_profile": {
    "company_name": "Partnerships Team",
    "signature": "Partnerships Team"
  },
  "selection_policy": "best_reviewed"
}


In [4]:
DEFAULT_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0 Safari/537.36"
    )
}


def slugify_url(url):
    parsed = urlparse(url)
    host = parsed.netloc.lower().replace("www.", "")
    path = parsed.path.strip("/").replace("/", "-")
    base = host if not path else f"{host}-{path}"
    base = re.sub(r"[^a-z0-9.-]+", "-", base.lower()).strip("-")
    return base or "storefront"


def fetch_html_file(url, output_dir):
    # This is the workflow entry point: fetch the live storefront HTML and save it as a file.
    response = requests.get(url, headers=DEFAULT_HEADERS, timeout=30)
    response.raise_for_status()
    html_text = response.text
    file_path = output_dir / f"{slugify_url(response.url)}.html"
    file_path.write_text(html_text, encoding="utf-8")
    return {
        "input_url": url,
        "final_url": response.url,
        "status_code": response.status_code,
        "html_path": str(file_path),
        "html_chars": len(html_text),
    }


In [5]:
FETCH_DIR = Path("html")
FETCH_DIR.mkdir(exist_ok=True)

def fetch_storefront_htmls(storefront_urls, output_dir):
    fetch_artifacts = []
    for url in storefront_urls:
        try:
            artifact = fetch_html_file(url, output_dir)
            artifact["fetch_status"] = "ok"
        except Exception as exc:
            artifact = {
                "input_url": url,
                "fetch_status": "error",
                "error": f"{type(exc).__name__}: {exc}",
            }
        fetch_artifacts.append(artifact)
    return fetch_artifacts


fetch_artifacts = fetch_storefront_htmls(storefront_urls, FETCH_DIR)

show_json(fetch_artifacts)


[
  {
    "input_url": "https://flybyjing.com/",
    "final_url": "https://flybyjing.com/",
    "status_code": 200,
    "html_path": "html/flybyjing.com.html",
    "html_chars": 351774,
    "fetch_status": "ok"
  },
  {
    "input_url": "https://www.allbirds.com/",
    "final_url": "https://www.allbirds.com/",
    "status_code": 200,
    "html_path": "html/allbirds.com.html",
    "html_chars": 989545,
    "fetch_status": "ok"
  }
]


## Part 1. Inspect the HTML Fetch Step

The lecture workflow starts with storefront HTML. Here the first task is a deterministic Python fetch step that turns each URL into a saved HTML file.


In [6]:
successful_fetches = [artifact for artifact in fetch_artifacts if artifact["fetch_status"] == "ok"]
if not successful_fetches:
    raise RuntimeError(
        "All storefront fetches failed. Check the input URLs or network access before continuing."
    )

print("Successful fetches:", len(successful_fetches))
for artifact in successful_fetches:
    print()
    print("input_url:", artifact["input_url"])
    print("final_url:", artifact["final_url"])
    print("html_path:", artifact["html_path"])
    print("html_chars:", artifact["html_chars"])


Successful fetches: 2

input_url: https://flybyjing.com/
final_url: https://flybyjing.com/
html_path: html/flybyjing.com.html
html_chars: 351774

input_url: https://www.allbirds.com/
final_url: https://www.allbirds.com/
html_path: html/allbirds.com.html
html_chars: 989545


In [7]:
if successful_fetches:
    sample_html_path = Path(successful_fetches[-1]["html_path"])
    sample_html = sample_html_path.read_text(encoding="utf-8", errors="ignore")
    print("Sample HTML file:", sample_html_path.name)
    print()
    print(sample_html[:1200])
else:
    print("No HTML file available because all fetches failed.")


Sample HTML file: allbirds.com.html

<!doctype html>
<html class='h-full' lang='en-US'>
  <head>
    <meta charset='utf-8'>
    <meta http-equiv='X-UA-Compatible' content='IE=edge'>
    <meta name='viewport' content='width=device-width,initial-scale=1'>
    <meta name='theme-color' content='#ece9e2'>
    <link rel='canonical' href='https://www.allbirds.com/'>
<meta name='description' content='Allbirds: The world’s most comfortable shoes, flats, and clothing made with natural materials like merino wool and eucalyptus. FREE shipping &amp; returns.'>
<title>Allbirds: Comfortable, Sustainable Shoes &amp; Apparel</title>


<meta property='og:site_name' content='Allbirds'>
<meta property='og:url' content='https://www.allbirds.com/'>
<meta property='og:title' content='The World’s Most Comfortable Shoes'>
<meta property='og:type' content='website'>
<meta property='og:description' content='The world’s most comfortable shoes, made with natural materials like merino wool, eucalyptus tree fiber, a

## Part 2. Extract Deterministic Storefront Signals

Before asking the model anything, we extract compact signals from the HTML file:
- page title;
- meta description;
- headings;
- navigation-like labels;
- visible text excerpt;
- Shopify evidence, if present.

This keeps the LLM prompts focused and makes debugging easier.


In [8]:
def unique_keep_order(items):
    seen = set()
    kept = []
    for item in items:
        key = item.strip()
        if not key or key in seen:
            continue
        seen.add(key)
        kept.append(key)
    return kept


def normalize_space(text):
    return re.sub(r"\s+", " ", text).strip()


def infer_store_name(og_site_name, title, final_url):
    if og_site_name:
        cleaned = normalize_space(og_site_name)
        if 1 <= len(cleaned.split()) <= 6:
            return cleaned
    if title:
        first_chunk = re.split(r"\s*[|:\-–]+\s*", title)[0].strip()
        if 1 <= len(first_chunk.split()) <= 6:
            return first_chunk
    host = urlparse(final_url).netloc.lower().replace("www.", "")
    host = host.split(".")[0]
    return host.replace("-", " ").title()


In [9]:
def extract_storefront_signals(html_path, final_url):
    # Deterministic parsing shrinks the raw HTML into a compact artifact for later tasks.
    html_text = Path(html_path).read_text(encoding="utf-8", errors="ignore")
    soup = BeautifulSoup(html_text, "html.parser")

    title = soup.title.get_text(" ", strip=True) if soup.title else ""
    og_site_name_tag = soup.find("meta", attrs={"property": "og:site_name"})
    og_site_name = og_site_name_tag.get("content", "").strip() if og_site_name_tag else ""
    meta_description_tag = soup.find("meta", attrs={"name": "description"})
    meta_description = meta_description_tag.get("content", "").strip() if meta_description_tag else ""

    for bad in soup(["script", "style", "noscript", "svg"]):
        bad.decompose()

    headings = unique_keep_order(
        normalize_space(tag.get_text(" ", strip=True))
        for tag in soup.find_all(["h1", "h2", "h3"])[:30]
    )

    nav_labels = unique_keep_order(
        normalize_space(tag.get_text(" ", strip=True))
        for tag in soup.find_all(["a", "button"])[:250]
        if 2 <= len(normalize_space(tag.get_text(" ", strip=True))) <= 40
    )[:25]

    visible_text_parts = [
        normalize_space(text)
        for text in soup.stripped_strings
        if len(normalize_space(text)) >= 20
    ]
    visible_text = " ".join(visible_text_parts)
    text_excerpt = visible_text[:5000]

    shopify_signals = unique_keep_order(
        signal
        for signal, present in {
            "cdn.shopify.com": "cdn.shopify.com" in html_text,
            ".myshopify.com": ".myshopify.com" in html_text,
            "Shopify.shop": "Shopify.shop" in html_text,
            "Powered by Shopify": "Powered by Shopify" in html_text,
        }.items()
        if present
    )

    return {
        "store_name": infer_store_name(og_site_name, title, final_url),
        "final_url": final_url,
        "html_path": str(html_path),
        "og_site_name": og_site_name,
        "page_title": title,
        "meta_description": meta_description,
        "headings": headings[:12],
        "nav_labels": nav_labels[:16],
        "text_excerpt": text_excerpt,
        "shopify_signals": shopify_signals,
    }


In [10]:
def build_storefront_inputs(successful_fetches):
    storefront_inputs = []
    for artifact in successful_fetches:
        signals = extract_storefront_signals(artifact["html_path"], artifact["final_url"])
        storefront_inputs.append(
            {
                "store_id": slugify_url(artifact["final_url"]),
                "input_url": artifact["input_url"],
                "final_url": artifact["final_url"],
                "html_path": artifact["html_path"],
                "signals": signals,
            }
        )
    return storefront_inputs


storefront_inputs = build_storefront_inputs(successful_fetches)

print("Prepared workflow inputs:", len(storefront_inputs))


Prepared workflow inputs: 2


In [11]:
for item in storefront_inputs:
    print(f"== {item['signals']['store_name']} ==")
    print("html_path:", item["html_path"])
    print("title:", item["signals"]["page_title"])
    print("meta:", item["signals"]["meta_description"])
    print("headings:", item["signals"]["headings"][:6])
    print("shopify_signals:", item["signals"]["shopify_signals"])
    print("text_excerpt:", short(item["signals"]["text_excerpt"], 500))
    print()


== FLY BY JING ==
html_path: html/flybyjing.com.html
title: Shop the Best Chinese Chili Sauce

        – FLY BY JING
meta: Meet the first premium Chinese chili sauce that’s changing the flavor game for good. Hot, spicy, crispy, tingly, and good on everything!
headings: ['6 MINUTES TO SICHUAN', 'Sauces', 'Noodles', 'Pantry', 'Merch', 'HEY, JING HERE.']
shopify_signals: ['cdn.shopify.com', '.myshopify.com', 'Shopify.shop', 'Powered by Shopify']
text_excerpt: Shop the Best Chinese Chili Sauce – FLY BY JING NEW CREAMY SESAME & ROASTED GARLIC NOODLES 🍜 FREE SHIPPING OVER $75 📦 GET 20% OFF & FREE SHIPPING ALWAYS⚡ 6 MINUTES TO SICHUAN Bring Sichuan street food flavors to your kitchen in 6 minutes with NEW Creamy Sesame and Roasted Garlic Noodles. Available only at Whole Foods and flybyjing.com. 6 MINUTES TO SICHUAN Bring Sichuan street food flavors to your kitchen in 6 minutes with NEW Creamy Sesame and Roasted Garlic Noodles. Available only at Whole ...

== Allbirds ==
html_path: html/allbir

## Part 3. Define the Workflow Contracts

The lecture emphasizes modular interfaces. We will keep the workflow contracts stable even though the outputs may still be uncertain.


In [12]:
SHOPIFY_WORKFLOW_GOAL = (
    "Starting from a storefront URL and fetched HTML file, infer the store positioning, "
    "propose a fitting Shopify plug-in idea, and draft one tailored outreach email."
)

WORKFLOW_SPEC = {
    "goal": SHOPIFY_WORKFLOW_GOAL,
    "topology": "DAG",
    "tasks": [
        "fetch_html_file",
        "extract_storefront_signals",
        "summarize_storefront",
        "generate_plugin_concept",
        "plan_email",
        "draft_email",
        "mock_send_email",
    ],
    "edges": [
        ("fetch_html_file", "extract_storefront_signals"),
        ("extract_storefront_signals", "summarize_storefront"),
        ("summarize_storefront", "generate_plugin_concept"),
        ("summarize_storefront", "plan_email"),
        ("generate_plugin_concept", "plan_email"),
        ("plan_email", "draft_email"),
        ("generate_plugin_concept", "draft_email"),
        ("draft_email", "mock_send_email"),
    ],
}

EMAIL_TASK_CONTRACT = {
    "input_fields": ["store_name", "concept_name", "concept", "rationale"],
    "output_fields": ["subject_line", "body"],
}

show_json(
    {
        "workflow_spec": WORKFLOW_SPEC,
        "email_task_contract": EMAIL_TASK_CONTRACT,
    }
)


{
  "workflow_spec": {
    "goal": "Starting from a storefront URL and fetched HTML file, infer the store positioning, propose a fitting Shopify plug-in idea, and draft one tailored outreach email.",
    "topology": "DAG",
    "tasks": [
      "fetch_html_file",
      "extract_storefront_signals",
      "summarize_storefront",
      "generate_plugin_concept",
      "plan_email",
      "draft_email",
      "mock_send_email"
    ],
    "edges": [
      [
        "fetch_html_file",
        "extract_storefront_signals"
      ],
      [
        "extract_storefront_signals",
        "summarize_storefront"
      ],
      [
        "summarize_storefront",
        "generate_plugin_concept"
      ],
      [
        "summarize_storefront",
        "plan_email"
      ],
      [
        "generate_plugin_concept",
        "plan_email"
      ],
      [
        "plan_email",
        "draft_email"
      ],
      [
        "generate_plugin_concept",
        "draft_email"
      ],
      [
        "draft_

## Part 4. Live LLM Helpers

We reuse the Lab 8 connection pattern: all reasoning-heavy tasks call `minimax-m2.5:cloud` through Ollama.


In [13]:
try:
    import ollama

    OLLAMA_IMPORT_ERROR = None
    OLLAMA_CLIENT = ollama.Client()
except Exception as exc:
    ollama = None
    OLLAMA_CLIENT = None
    OLLAMA_IMPORT_ERROR = repr(exc)


TARGET_OLLAMA_MODEL = "minimax-m2.5:cloud"


def get_ollama_status():
    # This check decides whether live inference should run or whether generic fallbacks should take over.
    if OLLAMA_CLIENT is None:
        return False, f"Python import failed: {OLLAMA_IMPORT_ERROR}"
    try:
        OLLAMA_CLIENT.ps()
        return True, "Connected to Ollama."
    except Exception as exc:
        return False, f"{type(exc).__name__}: {exc}"


OLLAMA_READY, OLLAMA_STATUS = get_ollama_status()
print("Target backend:", TARGET_OLLAMA_MODEL)
print("Ollama ready:", OLLAMA_READY)
print("Status:", OLLAMA_STATUS)


Target backend: minimax-m2.5:cloud
Ollama ready: True
Status: Connected to Ollama.


In [14]:
def extract_message_text(chat_response):
    data = to_builtin(chat_response)
    return (data.get("message") or {}).get("content")


def run_live_chat(messages, temperature=0):
    # Keep model transport separate from task logic so each task stays readable.
    if not OLLAMA_READY:
        return None, OLLAMA_STATUS
    try:
        response = OLLAMA_CLIENT.chat(
            model=TARGET_OLLAMA_MODEL,
            messages=messages,
            options={"temperature": temperature},
        )
        return to_builtin(response), None
    except Exception as exc:
        return None, f"{type(exc).__name__}: {exc}"


def extract_json_object(text):
    # Every task returns JSON so the workflow contract remains stable.
    if text is None:
        raise ValueError("No model text to parse.")
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned, flags=re.IGNORECASE)
        cleaned = re.sub(r"\s*```$", "", cleaned)
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        start = cleaned.find("{")
        end = cleaned.rfind("}")
        if start == -1 or end == -1 or start >= end:
            raise
        return json.loads(cleaned[start : end + 1])


def run_live_json_task(system_prompt, user_prompt, fallback, temperature=0):
    fallback_result = deepcopy(fallback)
    if isinstance(fallback_result, dict):
        fallback_result.setdefault("_source", "fallback")

    response, error = run_live_chat(
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
    )
    if error:
        if isinstance(fallback_result, dict):
            fallback_result["_fallback_reason"] = error
        return fallback_result

    text = extract_message_text(response)
    try:
        parsed = extract_json_object(text)
        if isinstance(parsed, dict):
            parsed["_source"] = "live"
        return parsed
    except Exception as exc:
        if isinstance(fallback_result, dict):
            fallback_result["_fallback_reason"] = f"JSON parse failed: {exc}"
            fallback_result["_raw_model_text"] = short(text, 280)
        return fallback_result


## Part 5. Generic Storefront Summary Task

This task accepts the fetched HTML artifact indirectly through the extracted signals.


In [15]:
STOPWORDS = {
    "the", "and", "for", "with", "your", "from", "that", "this", "shop", "all", "new",
    "our", "are", "you", "into", "more", "now", "sale", "best", "shopify", "store",
    "free", "shipping", "returns", "home", "main", "menu"
}


def infer_top_terms(signals, limit=6):
    text = " ".join(
        [
            signals.get("page_title", ""),
            signals.get("meta_description", ""),
            " ".join(signals.get("headings", [])),
            " ".join(signals.get("nav_labels", [])),
        ]
    ).lower()
    tokens = re.findall(r"[a-zA-Z][a-zA-Z-]{2,}", text)
    tokens = [token for token in tokens if token not in STOPWORDS]
    ranked = [token for token, _ in Counter(tokens).most_common()]
    return ranked[:limit]


def generic_list(values, default_label):
    cleaned = [value for value in values if value]
    return cleaned if cleaned else [default_label]


def normalize_sender_profile(sender_profile):
    # Sender details are configurable so the workflow output is not tied to one fixed company.
    data = dict(sender_profile or {})
    company_name = str(data.get("company_name") or "Partnerships Team").strip()
    signature = str(data.get("signature") or company_name).strip()
    return {
        "company_name": company_name,
        "signature": signature,
    }


In [16]:
def fallback_storefront_summary(storefront_input):
    # This fallback is generic: it derives cues from parsed HTML signals rather than from store-specific rules.
    signals = storefront_input["signals"]
    candidate_products = signals["headings"][:4] or signals["nav_labels"][:4]
    top_terms = infer_top_terms(signals)
    values = generic_list(top_terms[:4], "online retail")
    themes = generic_list(signals["headings"][:3], "category-led merchandising")
    evidence = generic_list(
        [signals["page_title"], signals["meta_description"], *signals["headings"][:2]],
        "visible storefront cues",
    )[:4]
    return {
        "store_id": storefront_input["store_id"],
        "store_name": signals["store_name"],
        "recipient_label": f"{signals['store_name']} team",
        "products": generic_list(candidate_products, "storefront categories"),
        "tone": generic_list(top_terms[4:8], "brand-led"),
        "values": values,
        "themes": themes,
        "evidence": evidence,
        "concise_summary": (
            f"{signals['store_name']} appears to focus on {', '.join(generic_list(candidate_products[:3], 'online products'))}. "
            f"The storefront emphasizes {', '.join(values[:3])} and presents itself through cues such as {', '.join(evidence[:2])}."
        ),
        "_source": "fallback",
    }


def normalize_summary(result, fallback, storefront_input):
    merged = deepcopy(fallback)
    if isinstance(result, dict):
        for key, value in result.items():
            if value not in (None, "", []):
                merged[key] = value
    for key in ["products", "tone", "values", "themes", "evidence"]:
        merged[key] = split_items(merged.get(key)) or fallback[key]
    merged["store_id"] = storefront_input["store_id"]
    merged["store_name"] = storefront_input["signals"]["store_name"]
    # Keep the greeting generic and reliable because the HTML rarely contains a real contact name.
    merged["recipient_label"] = f"{storefront_input['signals']['store_name']} team"
    merged["concise_summary"] = str(
        merged.get("concise_summary") or fallback["concise_summary"]
    ).strip()
    merged["_source"] = (
        result.get("_source", fallback["_source"]) if isinstance(result, dict) else fallback["_source"]
    )
    return merged


In [17]:
def summarize_storefront(storefront_input):
    fallback = fallback_storefront_summary(storefront_input)
    signals = storefront_input["signals"]
    user_prompt = dedent(
        f'''
        Storefront URL: {storefront_input["final_url"]}
        HTML file path: {storefront_input["html_path"]}

        Parsed storefront signals:
        {json.dumps(signals, indent=2)}

        Return JSON with exactly these keys:
        - store_id
        - store_name
        - recipient_label
        - products
        - tone
        - values
        - themes
        - evidence
        - concise_summary

        Keep the summary downstream-friendly and avoid inventing unavailable facts.
        '''
    ).strip()
    result = run_live_json_task(
        "You summarize one storefront for a Shopify workflow. Return JSON only.",
        user_prompt,
        fallback,
    )
    return normalize_summary(result, fallback, storefront_input)


In [18]:
def build_store_summaries(storefront_inputs):
    return {
        storefront_input["store_id"]: summarize_storefront(storefront_input)
        for storefront_input in storefront_inputs
    }


store_summaries = build_store_summaries(storefront_inputs)

for summary in store_summaries.values():
    print(f"== {summary['store_name']} ==")
    print("source:", summary["_source"])
    print("products:", ", ".join(summary["products"][:4]))
    print("values:", ", ".join(summary["values"][:4]))
    print("summary:", summary["concise_summary"])
    print()


== FLY BY JING ==
source: live
products: Premium Chinese chili sauces, Chili crisp/oil, Noodles (Creamy Sesame & Roasted Garlic), Pantry items
values: premium quality ingredients, authentic Sichuan cuisine, innovation in flavor, community building through membership
summary: Fly By Jing is a premium Chinese condiment brand specializing in authentic Sichuan-style chili sauces, chili crisp, and recently launched quick-cook noodles. Founded in 2018, the brand offers a membership program with 20% off and free shipping, emphasizing bold flavors, premium ingredients, and authentic Chinese culinary traditions delivered in convenient formats.

== Allbirds ==
source: live
products: Shoes (Men's, Women's), Apparel, Socks
values: sustainability, comfort, natural materials, eco-friendly
summary: Allbirds is a sustainable footwear and apparel brand offering comfortable shoes, socks, and clothing made from natural materials like merino wool, eucalyptus, and tree fiber. The brand emphasizes its commi

## Part 6. Generic Plug-in Concept Task

This task proposes one plug-in concept that fits the summarized storefront. 


In [19]:
def fallback_plugin_concept(summary):
    focus = summary["products"][0] if summary["products"] else "catalog discovery"
    theme = summary["themes"][0] if summary["themes"] else "storefront conversion"
    name_core = re.sub(r"[^A-Za-z0-9 ]+", " ", focus).strip().title() or "Storefront"
    return {
        "store_id": summary["store_id"],
        "name": f"{name_core} Guide",
        "concept": (
            f"An on-site Shopify plug-in that helps shoppers explore {focus.lower()} more clearly "
            f"and discover complementary items through a guided interface."
        ),
        "rationale": (
            f"The storefront already emphasizes {theme.lower()}, so a guided plug-in could turn "
            f"existing interest into a stronger shopping path."
        ),
        "value_to_store": (
            f"Helps {summary['store_name']} convert existing storefront signals into a more structured conversion flow."
        ),
        "evidence_from_summary": summary["evidence"][:3],
        "_source": "fallback",
    }


def normalize_concept(result, fallback, summary):
    merged = deepcopy(fallback)
    if isinstance(result, dict):
        for key, value in result.items():
            if value not in (None, "", []):
                merged[key] = value
    merged["store_id"] = summary["store_id"]
    merged["name"] = str(merged["name"]).strip()
    merged["concept"] = str(merged["concept"]).strip()
    merged["rationale"] = str(merged["rationale"]).strip()
    merged["value_to_store"] = str(merged["value_to_store"]).strip()
    merged["evidence_from_summary"] = split_items(merged.get("evidence_from_summary")) or fallback["evidence_from_summary"]
    merged["_source"] = (
        result.get("_source", fallback["_source"]) if isinstance(result, dict) else fallback["_source"]
    )
    return merged


In [20]:
def generate_plugin_concept(summary):
    fallback = fallback_plugin_concept(summary)
    prompt = dedent(
        f'''
        Store summary:
        {json.dumps(summary, indent=2)}

        Return JSON with exactly these keys:
        - store_id
        - name
        - concept
        - rationale
        - value_to_store
        - evidence_from_summary

        The plug-in should be plausible for a Shopify storefront and clearly tied to the summary evidence.
        '''
    ).strip()
    result = run_live_json_task(
        "You design one Shopify plug-in concept for a storefront. Return JSON only.",
        prompt,
        fallback,
        temperature=0.3,
    )
    return normalize_concept(result, fallback, summary)


In [21]:
def build_plugin_concepts(store_summaries):
    return {
        store_id: generate_plugin_concept(summary)
        for store_id, summary in store_summaries.items()
    }


plugin_concepts = build_plugin_concepts(store_summaries)

for concept in plugin_concepts.values():
    print(f"== {store_summaries[concept['store_id']]['store_name']} ==")
    print("source:", concept["_source"])
    print("plugin:", concept["name"])
    print("concept:", concept["concept"])
    print("rationale:", concept["rationale"])
    print()


== FLY BY JING ==
source: live
plugin: Sichuan Spice Explorer Quiz & Tastemaker Club Dashboard
concept: An interactive 'Spice Profile' quiz that segments customers by heat tolerance and flavor preferences, dynamically recommending products and pairing sauces with noodles. Integrated with the Tastemaker Club membership, the plug-in offers personalized 20% off discounts, early access to new drops, and exclusive 6-minute Sichuan noodle recipes. A 'Founder's Table' feature showcases Jing's Chengdu story to reinforce authentic origin, while 'Ingredient Transparency Badges' highlight premium quality to address the seed oil controversy blog content.
rationale: This plug-in directly addresses Fly By Jing's core value propositions: authentic Sichuan cuisine, premium ingredients, bold flavors, and community building. The Spice Profile Quiz solves the customer pain point of uncertain heat levels by guiding them to the right sauce, increasing conversion. The Tastemaker Club integration reinforces 

## Part 7. Generic Email Planning and Drafting Tasks

The lecture example separates planning from drafting. We keep the same shape:
1. decide the pitch strategy;
2. write the final email.


In [22]:
def fallback_email_plan(summary, concept):
    return {
        "store_id": summary["store_id"],
        "opening_hook": (
            f"What if {summary['store_name']} could make {summary['products'][0].lower()} easier to discover and buy?"
        ),
        "proof_points": [
            concept["rationale"],
            concept["value_to_store"],
            f"The storefront already signals {', '.join(summary['values'][:2])}.",
        ],
        "call_to_action": "Would you be open to a short demo if the idea looks relevant?",
        "tone_notes": "Keep the email specific, concise, and commercially useful.",
        "_source": "fallback",
    }


def normalize_email_plan(result, fallback, summary):
    merged = deepcopy(fallback)
    if isinstance(result, dict):
        for key, value in result.items():
            if value not in (None, "", []):
                merged[key] = value
    merged["store_id"] = summary["store_id"]
    merged["proof_points"] = split_items(merged.get("proof_points")) or fallback["proof_points"]
    for key in ["opening_hook", "call_to_action", "tone_notes"]:
        merged[key] = str(merged[key]).strip()
    if contains_placeholder_text(merged["call_to_action"]):
        merged["call_to_action"] = fallback["call_to_action"]
    merged["_source"] = (
        result.get("_source", fallback["_source"]) if isinstance(result, dict) else fallback["_source"]
    )
    return merged


In [23]:
def plan_email(summary, concept):
    fallback = fallback_email_plan(summary, concept)
    prompt = dedent(
        f'''
        Store summary:
        {json.dumps(summary, indent=2)}

        Plug-in concept:
        {json.dumps(concept, indent=2)}

        Return JSON with exactly these keys:
        - store_id
        - opening_hook
        - proof_points
        - call_to_action
        - tone_notes
        '''
    ).strip()
    result = run_live_json_task(
        "You plan a short outbound B2B email for a Shopify workflow. Return JSON only.",
        prompt,
        fallback,
    )
    return normalize_email_plan(result, fallback, summary)


In [24]:
def build_email_plans(store_summaries, plugin_concepts):
    return {
        store_id: plan_email(store_summaries[store_id], concept)
        for store_id, concept in plugin_concepts.items()
    }


email_plans = build_email_plans(store_summaries, plugin_concepts)

for plan in email_plans.values():
    print(f"== {store_summaries[plan['store_id']]['store_name']} ==")
    print("source:", plan["_source"])
    print("opening_hook:", plan["opening_hook"])
    print("call_to_action:", plan["call_to_action"])
    print()


== FLY BY JING ==
source: live
opening_hook: What if every customer who visited your store left knowing exactly which sauce to buy—and why they'd never settle for anything less? Fly By Jing's new Sichuan Spice Explorer Quiz turns uncertain shoppers into confident buyers, while the Tastemaker Club Dashboard turns first-time purchasers into lifelong members.
call_to_action: Let's talk about integrating the Spice Explorer Quiz & Tastemaker Club Dashboard into your platform. We're looking for partners who can help us scale personalized flavor recommendations and membership engagement to match our rapid growth.

== Allbirds ==
source: live
opening_hook: Hi Allbirds team — we love how you're turning sustainability into a measurable impact with your 'Sustainability In Every Step' mission. What if you could make that impact visible to every shopper on your product pages?
call_to_action: Would your team be open to a quick demo of how the Eco-Impact Dashboard could integrate with your product pa

In [25]:
def fallback_email_draft(summary, concept, plan, sender_profile):
    sender_profile = normalize_sender_profile(sender_profile)
    subject_line = f"Idea for {summary['store_name']}: {concept['name']}"
    body = dedent(
        f'''
        Hi {summary['recipient_label']},

        I reviewed {summary['store_name']} and noticed the storefront already emphasizes {', '.join(summary['values'][:2])}. That made me think of one plug-in idea that could fit naturally: {concept['name']}.

        We are exploring this through {sender_profile['company_name']}. {concept['name']} would {concept['concept'][0].lower() + concept['concept'][1:]} {concept['rationale']}

        The reason this looks relevant is simple: {concept['value_to_store']}

        {plan['call_to_action']}

        Best,
        {sender_profile['signature']}
        '''
    ).strip()
    return {
        "subject_line": subject_line,
        "body": body,
        "_source": "fallback",
    }


def normalize_email_draft(result, fallback):
    merged = deepcopy(fallback)
    if isinstance(result, dict):
        for key, value in result.items():
            if value not in (None, "", []):
                merged[key] = value
    merged["subject_line"] = str(merged["subject_line"]).strip()
    merged["body"] = str(merged["body"]).strip()
    merged["_source"] = (
        result.get("_source", fallback["_source"]) if isinstance(result, dict) else fallback["_source"]
    )
    return merged


In [26]:
def draft_email(summary, concept, plan, sender_profile):
    sender_profile = normalize_sender_profile(sender_profile)
    fallback = fallback_email_draft(summary, concept, plan, sender_profile)
    prompt = dedent(
        f'''
        Store summary:
        {json.dumps(summary, indent=2)}

        Plug-in concept:
        {json.dumps(concept, indent=2)}

        Email plan:
        {json.dumps(plan, indent=2)}

        Sender profile:
        {json.dumps(sender_profile, indent=2)}

        Return JSON with exactly these keys:
        - subject_line
        - body

        Constraints:
        - body must begin exactly with: Hi {summary['recipient_label']},
        - mention the storefront name {summary['store_name']} explicitly;
        - mention the plug-in name explicitly;
        - keep the email concise and commercially plausible;
        - end with one soft call to action in the final sentence;
        - do not include placeholder links or bracketed placeholder text;
        - no Markdown, no bullet lists.
        '''
    ).strip()
    result = run_live_json_task(
        "You write one tailored outreach email for a Shopify workflow. Return JSON only.",
        prompt,
        fallback,
    )
    return normalize_email_draft(result, fallback)


def mock_send_email(storefront_input, draft):
    # Keep the external side effect mocked while still preserving the workflow shape.
    return {
        "status": "mock_sent",
        "store_id": storefront_input["store_id"],
        "store_name": storefront_input["signals"]["store_name"],
        "store_url": storefront_input["final_url"],
        "html_path": storefront_input["html_path"],
        "subject_line": draft["subject_line"],
        "body_preview": short(draft["body"], 220),
    }


## Part 8. Orchestrate the Full Workflow

The orchestration code should stay short because the task logic already lives in the task functions.


In [27]:
def run_shopify_workflow_for_storefront(
    storefront_input,
    store_summaries,
    plugin_concepts,
    email_plans,
    sender_profile,
):
    summary = store_summaries[storefront_input["store_id"]]
    concept = plugin_concepts[storefront_input["store_id"]]
    plan = email_plans[storefront_input["store_id"]]
    draft = draft_email(summary, concept, plan, sender_profile)
    delivery = mock_send_email(storefront_input, draft)
    return {
        "storefront_input": storefront_input,
        "summary": summary,
        "concept": concept,
        "email_plan": plan,
        "email_draft": draft,
        "delivery": delivery,
    }


def build_workflow_results(
    storefront_inputs,
    store_summaries,
    plugin_concepts,
    email_plans,
    sender_profile,
):
    sender_profile = normalize_sender_profile(sender_profile)
    return {
        storefront_input["store_id"]: run_shopify_workflow_for_storefront(
            storefront_input,
            store_summaries,
            plugin_concepts,
            email_plans,
            sender_profile,
        )
        for storefront_input in storefront_inputs
    }

In [28]:
workflow_results = build_workflow_results(
    storefront_inputs,
    store_summaries,
    plugin_concepts,
    email_plans,
    sender_profile,
)

for result in workflow_results.values():
    print(f"== {result['summary']['store_name']} ==")
    print("subject:", result["email_draft"]["subject_line"])
    print(
        "sources:",
        "summary=" + result["summary"]["_source"],
        "concept=" + result["concept"]["_source"],
        "plan=" + result["email_plan"]["_source"],
        "email=" + result["email_draft"]["_source"],
    )
    print("delivery:", result["delivery"]["status"])
    print()


== FLY BY JING ==
subject: Turn Every Visitor Into a Confident Buyer With FLY BY JING's New Spice Explorer Quiz
sources: summary=live concept=live plan=live email=live
delivery: mock_sent

== Allbirds ==
subject: Make Your Sustainability Impact Visible to Every Shopper
sources: summary=live concept=live plan=live email=live
delivery: mock_sent



## Part 9. Review the Outputs

A workflow is not educational if it only says “done.” We should inspect whether the outputs are usable, and allow uncertainty when the model produces something imperfect.


In [29]:
def review_workflow_result(result):
    issues = []
    summary = result["summary"]
    concept = result["concept"]
    draft = result["email_draft"]
    body = draft["body"].lower()
    final_sentence = re.split(r"(?<=[.!?])\s+", draft["body"].strip())[-1].lower()
    expected_greeting = f"hi {summary['recipient_label'].lower()},"

    if len(summary["concise_summary"].split()) < 16:
        issues.append("Summary may be too short to guide downstream tasks.")
    if not body.startswith(expected_greeting):
        issues.append("Email does not start with the expected recipient greeting.")
    if summary["store_name"].lower() not in body:
        issues.append("Email body does not mention the storefront name.")
    if concept["name"].split()[0].lower() not in body:
        issues.append("Email body does not mention the plug-in name.")
    if contains_placeholder_text(draft["body"]):
        issues.append("Email still contains placeholder text.")
    if not any(
        phrase in final_sentence
        for phrase in ["would", "open", "happy", "if useful", "let me know", "demo", "interested", "reply", "chat"]
    ):
        issues.append("Final sentence does not clearly function as a soft CTA.")

    return {
        "store_id": summary["store_id"],
        "store_name": summary["store_name"],
        "status": "PASS" if not issues else "REVIEW",
        "issues": issues,
        "subject_line": draft["subject_line"],
        "live_steps": sum(
            1
            for artifact in [
                result["summary"],
                result["concept"],
                result["email_plan"],
                result["email_draft"],
            ]
            if artifact.get("_source") == "live"
        ),
    }


In [30]:
review_report = [review_workflow_result(result) for result in workflow_results.values()]


def score_review_result(result, review):
    # Prefer reviewed passes, then richer live outputs, then more detailed summaries.
    pass_bonus = 100 if review["status"] == "PASS" else 0
    issue_penalty = 15 * len(review["issues"])
    live_bonus = 5 * review["live_steps"]
    detail_bonus = min(len(result["summary"]["concise_summary"].split()), 40)
    return pass_bonus + live_bonus + detail_bonus - issue_penalty


def select_store_for_final_email(workflow_results, review_report, selection_policy="best_reviewed"):
    if not workflow_results:
        raise ValueError("workflow_results is empty. The workflow must process at least one storefront.")
    review_by_store = {item["store_id"]: item for item in review_report}
    ranking = []
    for store_id, result in workflow_results.items():
        review = review_by_store[store_id]
        ranking.append(
            {
                "store_id": store_id,
                "store_name": result["summary"]["store_name"],
                "status": review["status"],
                "score": score_review_result(result, review),
                "live_steps": review["live_steps"],
                "issues": review["issues"],
            }
        )

    ranking.sort(key=lambda item: (-item["score"], item["store_name"].lower()))

    if selection_policy == "first":
        return next(iter(workflow_results)), ranking
    return ranking[0]["store_id"], ranking


selected_store_id, selection_ranking = select_store_for_final_email(
    workflow_results,
    review_report,
    selection_policy=selection_policy,
)

show_json(
    {
        "review_report": review_report,
        "selection_ranking": selection_ranking,
        "selected_store_id": selected_store_id,
    }
)


{
  "review_report": [
    {
      "store_id": "flybyjing.com",
      "store_name": "FLY BY JING",
      "status": "PASS",
      "issues": [],
      "subject_line": "Turn Every Visitor Into a Confident Buyer With FLY BY JING's New Spice Explorer Quiz",
      "live_steps": 4
    },
    {
      "store_id": "allbirds.com",
      "store_name": "Allbirds",
      "status": "PASS",
      "issues": [],
      "subject_line": "Make Your Sustainability Impact Visible to Every Shopper",
      "live_steps": 4
    }
  ],
  "selection_ranking": [
    {
      "store_id": "allbirds.com",
      "store_name": "Allbirds",
      "status": "PASS",
      "score": 160,
      "live_steps": 4,
      "issues": []
    },
    {
      "store_id": "flybyjing.com",
      "store_name": "FLY BY JING",
      "status": "PASS",
      "score": 160,
      "live_steps": 4,
      "issues": []
    }
  ],
  "selected_store_id": "allbirds.com"
}


## Part 10. Select One Store and Print the Final Email

The final output of the workflow is the email itself. The selection below is driven by `selection_policy`:
- `best_reviewed`: prefer the strongest reviewed output;
- `first`: keep the first successfully processed storefront.


In [31]:
selected_result = workflow_results[selected_store_id]

show_json(
    {
        "selected_store_id": selected_store_id,
        "store_name": selected_result["summary"]["store_name"],
        "store_url": selected_result["storefront_input"]["final_url"],
        "html_path": selected_result["storefront_input"]["html_path"],
        "plugin_name": selected_result["concept"]["name"],
        "subject_line": selected_result["email_draft"]["subject_line"],
    }
)


{
  "selected_store_id": "allbirds.com",
  "store_name": "Allbirds",
  "store_url": "https://www.allbirds.com/",
  "html_path": "html/allbirds.com.html",
  "plugin_name": "Eco-Impact Dashboard",
  "subject_line": "Make Your Sustainability Impact Visible to Every Shopper"
}


In [32]:
print("Store:", selected_result["summary"]["store_name"])
print("Storefront:", selected_result["storefront_input"]["final_url"])
print("HTML file:", selected_result["storefront_input"]["html_path"])
print("Subject:", selected_result["email_draft"]["subject_line"])
print()
print(selected_result["email_draft"]["body"])


Store: Allbirds
Storefront: https://www.allbirds.com/
HTML file: html/allbirds.com.html
Subject: Make Your Sustainability Impact Visible to Every Shopper

Hi Allbirds team,

We love how you're turning sustainability into a measurable impact with your 'Sustainability In Every Step' mission. What if you could make that impact visible to every shopper on your product pages?

The Eco-Impact Dashboard would display real-time carbon footprint savings, show how your natural materials like merino wool, eucalyptus, and tree fiber replace petroleum-based synthetics, and quantify environmental benefits like water saved and carbon offset per purchase. This transforms your 'Materials From The Earth' messaging into tangible, visible metrics that shoppers can see right where they're making their purchasing decision.

Beyond the product page, the dashboard creates shareable impact stats that customers can post on social media, amplifying your climate responsibility message organically. It reinforces w

In [33]:
show_json(selected_result["delivery"])


{
  "status": "mock_sent",
  "store_id": "allbirds.com",
  "store_name": "Allbirds",
  "store_url": "https://www.allbirds.com/",
  "html_path": "html/allbirds.com.html",
  "subject_line": "Make Your Sustainability Impact Visible to Every Shopper",
  "body_preview": "Hi Allbirds team, We love how you're turning sustainability into a measurable impact with your 'Sustainability In Every Step' mission. What if you could make that impact visible to every shopper on your product pages?..."
}


### <b style="color:orange">Task 1: Expand the Storefront List</b>

Add another public storefront URL that you are interested in to the `WORKFLOW_INPUTS['storefront_urls']` list at the beginning of the notebook. Then, run the entire workflow again to see how the system processes the new URL, extracts signals, and generates a tailored outreach email. 

This task helps you understand how the generic workflow adapts to different storefront contexts and content structures. Inspect the intermediate JSON artifacts to see the model's reasoning for your specific store.

Provide your chosen URL and briefly discuss the generated plug-in concept below.

**Discussion:**

## Key Takeaways

1. A lecture-style workflow becomes much more reusable when the input is a URL and a fetched HTML file rather than manually curated store notes.
2. Deterministic extraction and stable JSON contracts make the uncertain LLM steps easier to inspect.
3. The workflow can stay generic even when the outputs vary across storefronts.
4. Imperfect outputs are not a failure of the tutorial; they are part of what makes workflow inspection necessary.
5. The final output of the workflow is the drafted email, but the intermediate artifacts are what make the pipeline understandable and debuggable.


------

### <b style="color:orange">Submission</b>

Make sure you:
1. execute the notebook from top to bottom;
2. complete each task;
3. submit your finished notebook as `YourName_YourStudentID.ipynb` on Moodle.